## XGBoost


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

import warnings
from builtins import FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
'''
Load dataset and preprocessing
-> train | test | submission | prediction
'''

# Data preprocessing function
def preprocess_data(data):
    # ['date'] -> datetime
    data['date'] = pd.to_datetime(data['date'], format='%Y-%m-%d')
    # ordinal date feature
    data['date_ordinal'] = data['date'].map(datetime.toordinal)
    # store_menu_id
    data['store_menu_id'] = data['store'] + "_" + data['menu']

    return data

# train data
train_data = pd.read_csv('./dataset/train/train.csv')
train_data = preprocess_data(train_data)

# test data
for i in range(0, 10):
    test = pd.read_csv(f"./dataset/test/TEST_0{i}.csv")
    test = preprocess_data(test)
    # test_data_{i} for all test datasets
    globals()[f'test_data_{i}'] = test

# submission format
submission = pd.read_csv("./result/sample_submission_date.csv")

# Prediction result
all_preds = []


In [28]:
import csv

# 매장별 메뉴 딕셔너리
store_menus = {
    "라그로타": ['빵 추가 (1인)', '한우 (200g)', '콜라', '버섯 크림 리조또', '해산물 토마토 스튜 파스타', '모둠 해산물 플래터', '스프라이트', '카스', 'G-Charge(3)', '시저 샐러드 ', '아메리카노', '하이네켄(생)', 'Gls.Sileni', 'Gls.미션 서드', '양갈비 (4ps)', '그릴드 비프 샐러드', '자몽리치에이드', '까르보나라', 'AUS (200g)', '해산물 토마토 스파게티', '미션 서드 카베르네 쉬라', '해산물 토마토 리조또', 'Open Food', '알리오 에 올리오 ', '제로콜라'],
    "화담숲주막": ['참살이 막걸리', '해물파전', '스프라이트', '찹쌀식혜', '콜라', '병천순대', '느린마을 막걸리', '단호박 식혜 '],
    "연회장": ['Conference L2', 'Convention Hall', '매콤 무뼈닭발&계란찜', 'Conference M9', '모둠 돈육구이(3인)', '왕갈비치킨', '야채추가', 'Conference M8', '로제 치즈떡볶이', '공깃밥', 'Regular Coffee', 'OPUS 2', '주먹밥 (2ea)', 'Conference L3', '삼겹살추가 (200g)', '마라샹궈', 'Grand Ballroom', '골뱅이무침', '돈목살 김치찌개 (밥포함)', 'Cass Beer', 'Conference M1', 'Cookie Platter', 'Conference L1'],
    "포레스트릿": ['떡볶이', '스프라이트', '카페라떼(HOT)', '아메리카노(ICE)', '페스츄리 소시지', '생수', '코카콜라', '치즈 핫도그', '복숭아 아이스티', '카페라떼(ICE)', '아메리카노(HOT)', '꼬치어묵'],
    "화담숲카페": ['아메리카노 HOT', '메밀미숫가루', '카페라떼 ICE', '현미뻥스크림', '아메리카노 ICE'],
    "담하": ['메밀면 사리', '갑오징어 비빔밥', ' 한우 불고기 정식', '더덕 한우 지짐', '(후식) 물냉면', '제로콜라', '한우 떡갈비 정식', '(후식) 된장찌개', '스프라이트', '문막 복분자 칵테일', '(정식) 된장찌개', '한우 미역국 정식', '황태해장국', '(정식) 물냉면 ', '봉평메밀 물냉면', '콜라', '(단체) 황태해장국 3/27까지', ' 한우 불고기', '(정식) 비빔냉면', '라면사리', '(후식) 비빔냉면', '참이슬', '(단체) 한우 우거지 국밥', '(단체) 공깃밥', '카스', '한우 차돌박이 된장찌개', '테라', '꼬막 비빔밥', '명태회 비빔냉면', '은이버섯 갈비탕', '들깨 양지탕', '룸 이용료', '갱시기', '생목살 김치찌개', '(단체) 은이버섯 갈비탕', '느린마을 막걸리', '한우 우거지 국밥', '처음처럼', '공깃밥', '명인안동소주', '하동 매실 칵테일', '(단체) 생목살 김치전골 2.0'],
    "미라시아": ['콥 샐러드', '애플망고 에이드', '레인보우칵테일(알코올)', '공깃밥', '코카콜라(제로)', '오븐구이 윙과 킬바사소세지', '보일링 랍스타 플래터(덜매운맛)', '유자 하이볼', 'BBQ 고기추가', '핑크레몬에이드', 'BBQ Platter', '파스타면 추가(150g)', ' 브런치 (패키지)', '버드와이저(무제한)', '(오븐) 하와이안 쉬림프 피자', '브런치(대인) 주말', '브런치(어린이)', '칠리 치즈 프라이', '브런치(대인) 주중', '브런치 4인 패키지 ', '잭 애플 토닉', '(단체)브런치주중 36,000', '스텔라(무제한)', '보일링 랍스타 플래터', '쉬림프 투움바 파스타', '글라스와인 (레드)', '(화덕) 불고기 페퍼로니 반반피자', '브런치 2인 패키지 ', '코카콜라', '스프라이트', '얼그레이 하이볼'],
    "카페테리아": ['짬뽕밥', '샷 추가', '카페라떼(ICE)', '공깃밥(추가)', '약 고추장 돌솥비빔밥', '짬뽕', '단체식 18000(신)', '수제 등심 돈까스', '아메리카노(ICE)', '진사골 설렁탕', '치즈돈까스', '카페라떼(HOT)', '짜장면', '짜장밥', '오픈푸드', '어린이 돈까스', '단체식 13000(신)', '복숭아 아이스티', '구슬아이스크림', '한상 삼겹구이 정식(2인) 소요시간 약 15~20분', '새우 볶음밥', '아메리카노(HOT)', '새우튀김 우동', '돼지고기 김치찌개'],
    "느티나무 셀프BBQ": ['일회용 소주컵', '콜라 (단체)', '허브솔트', '참이슬 (단체)', '대여료 90,000원', '일회용 종이컵', '쌈장', '잔디그늘집 의자 추가', '카스 병(단체)', 'BBQ55(단체)', '친환경 접시 14cm', '신라면', '스프라이트 (단체)', '1인 수저세트', '잔디그늘집 대여료 (12인석)', '대여료 30,000원', '본삼겹 (단품,실내)', '햇반', '쌈야채세트', '친환경 접시 23cm', '잔디그늘집 대여료 (6인석)', '육개장 사발면', '대여료 60,000원']
}

# CSV로 저장
with open("./dataset/store_menus.csv", "w", newline='', encoding='utf-8-sig') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Store", "Menu"])
    for store, menus in store_menus.items():
        for menu in menus:
            writer.writerow([store, menu])
